# Manufacturing Data Engineering Exercise
Raw data pulled live from the exercise API. Every value is shown exactly as the API returned it, written as JSON, and rows are in the order the API sent them.

- `"text"` in quotes is a string, so `"1180"` is text while `1180` is a number
- `null` means the API sent an empty value
- a blank cell means the record didn't include that field at all

The only change is unwrapping the extra list each response comes in (`[[{...}, {...}]]`). Each response is also saved unchanged to `data/raw/` for the next notebooks.

In [25]:
import json
from pathlib import Path

import pandas as pd
import requests

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)


def fetch(endpoint):
    resp = requests.get(f"https://exercise.convanalytics.com/api/{endpoint}", timeout=30)
    resp.raise_for_status()
    (RAW_DIR / f"{endpoint}.json").write_bytes(resp.content)
    # The API wraps the records in an extra list: [[{...}, {...}]]
    return [record for page in resp.json() for record in page]


def show(records):
    # Show each value as its raw JSON so types, nulls and stray spaces stay visible.
    table = pd.DataFrame(
        [{key: json.dumps(value) for key, value in record.items()} for record in records]
    ).fillna("")
    left = [("font-family", "monospace"), ("text-align", "left"), ("white-space", "pre")]
    return (
        table.style.hide(axis="index")
        .set_caption(f"{len(records)} records")
        .set_table_styles([{"selector": "th, td", "props": left}])
    )


production_runs = fetch("production_runs")
counter_readings = fetch("counter_readings")
downtime_events = fetch("downtime_events")

## Production runs
Production-run windows, products, equipment, and target rates.

In [26]:
show(production_runs)

run_id,equipment_id,start_time,end_time,product_code,target_rate,target_rate_unit
"""R100""","""PKG-01""","""2026-08-13T08:00:00-05:00""","""2026-08-13T09:00:00-05:00""","""FLOUR-A""",600,"""cases/hour"""
"""R101""","""PKG-01""","""2026-08-13T09:10:00-05:00""","""2026-08-13T10:10:00-05:00""","""flour-a """,10,"""cases/minute"""
"""R102""","""PKG-02""","""2026-08-13T08:15:00-05:00""","""2026-08-13T09:15:00-05:00""","""FLOUR-B""",420,"""cases/hour"""
"""R103""","""PKG-02""","""2026-08-13T09:30:00-05:00""","""2026-08-13T10:30:00-05:00""",""" FLOUR-B""",7,"""cases/minute"""
"""R104""","""PKG-01""","""2026-08-13T10:20:00-05:00""",null,"""FLOUR-C""",540,"""cases/hour"""


## Counter readings
Timestamped cumulative case-counter readings with a JSON payload.

In [27]:
show(counter_readings)

reading_id,equipment_id,recorded_at,payload
"""C001""","""PKG-01""","""2026-08-13T07:59:00-05:00""","{""count"": 1000, ""unit"": ""cases""}"
"""C002""","""PKG-01""","""2026-08-13T08:10:00-05:00""","{""count"": 1090, ""unit"": ""cases""}"
"""C003""","""PKG-01""","""2026-08-13T08:20:00-05:00""","{""count"": ""1180"", ""unit"": ""cases""}"
"""C004""","""PKG-01""","""2026-08-13T08:20:00-05:00""","{""count"": ""1180"", ""unit"": ""cases""}"
"""C005""","""PKG-01""","""2026-08-13T08:40:00-05:00""","{""count"": ""ERROR"", ""unit"": ""cases""}"
"""C006""","""PKG-01""","""2026-08-13T14:00:00Z""","{""count"": 1510, ""unit"": ""cases""}"
"""C007""","""PKG-01""","""2026-08-13T09:10:00-05:00""","{""count"": 1510, ""unit"": ""cases""}"
"""C008""","""PKG-01""","""2026-08-13T09:30:00-05:00""","{""count"": 1680, ""unit"": ""cases""}"
"""C009""","""PKG-01""","""2026-08-13T09:40:00-05:00""","{""count"": 1760, ""unit"": ""cases""}"
"""C010""","""PKG-01""","""2026-08-13T09:50:00-05:00""","{""count"": 25, ""unit"": ""cases""}"


## Downtime events
Timestamped equipment downtime intervals and recorded reasons.## Observations
- `production_runs`
  - non-uniform `product_code` and `target_rate_unit`
  - should have a `unit` column and `target_rate_per_hour`
- `counter_readings`
  - `payload` can be split into multiple columns
- `downtime_events`
  - reasons are vague
- all tables: times should be standardized (usually UTC, `+00:00` / `Z` timezone)

In [28]:
show(downtime_events)

event_id,equipment_id,start_time,end_time,reason
"""D001""","""PKG-01""","""2026-08-13T08:30:00-05:00""","""2026-08-13T08:36:00-05:00""","""Material unavailable"""
"""D002""","""PKG-01""","""2026-08-13T09:08:00-05:00""","""2026-08-13T09:14:00-05:00""","""Unknown"""
"""D003""","""PKG-01""","""2026-08-13T09:43:30-05:00""","""2026-08-13T09:47:30-05:00""","""material"""
"""D004""","""PKG-01""","""2026-08-13T09:46:00-05:00""","""2026-08-13T09:49:00-05:00""","""MATL"""
"""D005""","""PKG-02""","""2026-08-13T08:40:00-05:00""","""2026-08-13T08:45:00-05:00""","""Jam"""
"""D006""","""PKG-02""","""2026-08-13T09:48:00-05:00""","""2026-08-13T09:49:00-05:00""","""Unknown"""
"""D007""","""PKG-02""","""2026-08-13T09:53:20-05:00""","""2026-08-13T09:54:10-05:00""","""Unknown"""
"""D008""","""PKG-02""","""2026-08-13T09:58:05-05:00""","""2026-08-13T09:59:00-05:00""","""Unknown"""
"""D009""","""PKG-02""","""2026-08-13T10:03:40-05:00""","""2026-08-13T10:04:30-05:00""","""Unknown"""
"""D010""","""PKG-01""","""2026-08-13T10:34:00-05:00""",null,"""Operator stop"""


## Observations
- `production_runs`
  - non-uniform `product_code` and `target_rate_unit`
  - should have a `unit` column and `target_rate_per_hour`
- `counter_readings`
  - `payload` can be split into multiple columns
- `downtime_events`
  - reasons are vague
- all tables: times should be standardized (usually UTC, `+00:00` / `Z` timezone)